In [ ]:
#!pip install optunity
from google.colab import drive
#drive.mount('/content/drive')
import pandas as pd

# O caminho para o seu arquivo no Drive
caminho = '/content/drive/MyDrive/acarau.txt'

# Lendo o arquivo com os ajustes necessários para o formato da FUNCEME
df = pd.read_csv(caminho, sep=';', encoding='latin1')

# Visualizar as primeiras 5 linhas
#df.head()
#df.groupby('Anos')['Total'].sum().plot(kind='bar', figsize=(12,5))
media_historica = df.groupby('Anos')['Total'].sum().mean()
#print(f"A chuva média anual em Acaraú é de: {media_historica:.2f} mm")

# Gerando o gráfico com a linha da média
#ax = df.groupby('Anos')['Total'].sum().plot(kind='bar', figsize=(15,6), color='skyblue')
#ax.axhline(media_historica, color='red', linestyle='--', label=f'Média ({media_historica:.0f} mm)')
#ax.legend()
limite_seca = media_historica * 0.7
anos_totais = df.groupby('Anos')['Total'].sum()
anos_seca = anos_totais[anos_totais < limite_seca]

#print(f"Anos de seca severa (abaixo de {limite_seca:.0f} mm):")
#print(anos_seca)

import matplotlib.pyplot as plt

# Calculando a média para cada mês
media_mensal = df.groupby('Meses')['Total'].mean()

# Criando o gráfico
#plt.figure(figsize=(10,6))
#media_mensal.plot(kind='bar', color='teal', edgecolor='black')

# Perfumaria (Títulos e Rótulos)
#plt.title('Média Mensal de Precipitação em Acaraú (1974-2026)', fontsize=14)
#plt.xlabel('Mês', fontsize=12)
#plt.ylabel('Precipitação Média (mm)', fontsize=12)
#plt.xticks(range(0, 12), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'], rotation=45)
#plt.grid(axis='y', linestyle='--', alpha=0.7)
#plt.show()
# Filtrando apenas os meses da quadra chuvosa
quadra = df[df['Meses'].isin([2, 3, 4, 5])]
chuva_quadra = quadra['Total'].sum()
chuva_total_historica = df['Total'].sum()

porcentagem = (chuva_quadra / chuva_total_historica) * 100

#print(f"Em Acaraú, {porcentagem:.2f}% de toda a chuva do ano cai entre Fevereiro e Maio.")

# 1. Calculando a porcentagem da Quadra Chuvosa (Fev a Maio)
quadra_meses = [2, 3, 4, 5]
chuva_quadra = df[df['Meses'].isin(quadra_meses)]['Total'].sum()
chuva_total_geral = df['Total'].sum()

porcentagem_geral = (chuva_quadra / chuva_total_geral) * 100

print(f"--- ANÁLISE DE CONCENTRAÇÃO ---")
print(f"Em Acaraú, {porcentagem_geral:.2f}% de toda a chuva histórica cai entre Fevereiro e Maio.")

# 2. Evolução por Década (para ver se o clima está mudando)
df['Decada'] = (df['Anos'] // 10) * 10
analise_decada = df.groupby('Decada').apply(
    lambda x: (x[x['Meses'].isin(quadra_meses)]['Total'].sum() / x['Total'].sum()) * 100
)

#print("\n--- PORCENTAGEM DA QUADRA POR DÉCADA ---")
#print(analise_decada)
!pip install pymannkendall
import pymannkendall as mk
import numpy as np
import matplotlib.pyplot as plt

# 1. Preparando os dados (removendo 2026 para não enviesar a tendência)
dados_anuais = df[df['Anos'] < 2026].groupby('Anos')['Total'].sum()
anos = dados_anuais.index.values
valores = dados_anuais.values

# 2. Cálculos: Mann-Kendall e Estimador de Sen
res_mk = mk.original_test(valores)
slope = res_mk.slope
intercept = res_mk.intercept
# Criando a linha de tendência: y = ax + b
reta_tendencia = intercept + slope * (anos - anos[0])

# 3. Cálculos: Pettitt (Ponto de Quebra)
def pettitt_test(data):
    n = len(data); k = range(n); u = np.zeros(n)
    for i in k: u[i] = np.sum(np.sign(data[i] - data))
    s = np.cumsum(u); cp = np.where(np.abs(s) == np.max(np.abs(s)))[0][0]
    return cp
indice_quebra = pettitt_test(valores)
ano_quebra = anos[indice_quebra]
media_antes = np.mean(valores[:indice_quebra])
media_depois = np.mean(valores[indice_quebra:])

# --- GRÁFICO FINAL ---
#plt.figure(figsize=(15, 7))

# Curva experimental (Dados Reais)
#plt.plot(anos, valores, marker='o', color='lightgray', alpha=0.6, label='Chuva Anual (Dados Reais)')

# 1. Reta de Mann-Kendall (AZUL E TRACEJADA)
#plt.plot(anos, reta_tendencia, color='blue', linestyle='--', linewidth=2,
#label=f'Tendência MK/Sen (Slope: {slope:.2f} mm/ano)')

# 2. Pettitt (VERMELHO COM DEGRAU)
#plt.hlines(media_antes, anos[0], ano_quebra, colors='red', linestyles='--', linewidth=2, label='Média Pettitt (Pré-quebra)')
#plt.hlines(media_depois, ano_quebra, anos[-1], colors='red', linestyles='--', linewidth=2, label='Média Pettitt (Pós-quebra)')
#plt.vlines(ano_quebra, media_antes, media_depois, colors='red', linestyles='--', linewidth=2) # O Degrau

# Formatação Adicional
#plt.axvline(ano_quebra, color='red', alpha=0.2, linestyle=':')
#plt.title(f'Análise Estocástica de Acaraú\nMann-Kendall (Azul) vs Pettitt (Vermelho)', fontsize=14)
#plt.xlabel('Ano')
#plt.ylabel('Precipitação Total (mm)')
#plt.legend(loc='upper left')
#plt.grid(alpha=0.3)

#plt.show()

#print(f"Inclinação da reta (Slope): {slope:.2f} mm/ano")
#print(f"Resultado MK: {res_mk.trend}")

import pymannkendall as mk

# Preparando os dados (1974-2025)
dados_anuais = df[df['Anos'] < 2026].groupby('Anos')['Total'].sum()
valores = dados_anuais.values

# Executando o teste completo
resultado = mk.original_test(valores)

# Extraindo os valores
slope = resultado.slope
p_valor = resultado.p

#print(f"--- VEREDITO DO ESTIMADOR DE SEN ---")
#print(f"Inclinação (Slope): {slope:.4f} mm/ano")
#print(f"Significância (p-value): {p_valor:.4f}")

#if slope > 0:
    #print(f"\nA tendência indica um AUMENTO de {slope:.2f} mm a cada ano.")
#elif slope < 0:
    #print(f"\nA tendência indica uma DIMINUIÇÃO de {abs(slope):.2f} mm a cada ano.")
#else:
    #print("\nA tendência é perfeitamente neutra.")

#if p_valor > 0.05:
    #print("Nota: Como o p-value é alto, essa variação é considerada estatisticamente insignificante (estabilidade).")
from sklearn.model_selection import train_test_split

# 1. Removendo 2026 para evitar dados incompletos no treino
df_ml = df[df['Anos'] < 2026].copy()

# 2. Selecionando as 'Features' (X) e o 'Alvo' (y)
# Vamos tentar prever o Total (y) usando o Ano e o Mês (X)
X = df_ml[['Anos', 'Meses']]
y = df_ml['Total']

# 3. Divisão Temporal (Train-Test Split)
# Em séries temporais, evite o 'shuffle=True'. Queremos os últimos anos para teste.
# Vamos separar 20% para teste (os anos mais recentes)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

#print(f"Dados de Treino: {len(X_train)} registros")
#print(f"Dados de Teste: {len(X_test)} registros")
#print(f"Período de Teste: De {X_test['Anos'].min()} a {X_test['Anos'].max()}")

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# 1. Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# 2. XGBoost
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

# 3. Avaliação
mae_rf = mean_absolute_error(y_test, rf_pred)
mae_xgb = mean_absolute_error(y_test, xgb_pred)

#print(f"--- PERFORMANCE DOS MODELOS ---")
#print(f"MAE Random Forest: {mae_rf:.2f} mm")
#print(f"MAE XGBoost: {mae_xgb:.2f} mm")

import matplotlib.pyplot as plt

# Calculando os resíduos (Real - Previsto)
residuos_rf = y_test - rf_pred
residuos_xgb = y_test - xgb_pred

#plt.figure(figsize=(15, 6))

# Linha de referência Zero (Perfeição)
#plt.axhline(0, color='black', linestyle='-', linewidth=1.5, label='Erro Zero (Referência)')

# Plotando os erros
#plt.scatter(range(len(residuos_rf)), residuos_rf, color='blue', alpha=0.5, label='Erro Random Forest')
#plt.scatter(range(len(residuos_xgb)), residuos_xgb, color='red', marker='x', alpha=0.5, label='Erro XGBoost')

# Preenchendo a área do "Desvio Médio" (MAE) para o Random Forest como exemplo
#plt.fill_between(range(len(residuos_rf)), -mae_rf, mae_rf, color='blue', alpha=0.1, label='Banda de Confiança (MAE)')

#plt.title('Análise de Resíduos: Onde os modelos se afastam da Realidade (Acaraú)')
#plt.xlabel('Meses do Período de Teste')
#plt.ylabel('Diferença (Real - Previsto) em mm')
#plt.legend()
#plt.grid(alpha=0.3)
#plt.show()

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

# Correlação de Pearson
corr_rf, _ = pearsonr(y_test, rf_pred)
corr_xgb, _ = pearsonr(y_test, xgb_pred)

# RMSE (Raiz do Erro Quadrático Médio)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb_pred))

#print(f"--- ALINHAMENTO GEOMÉTRICO (Pearson) ---")
#print(f"Random Forest: {corr_rf:.4f}")
#print(f"XGBoost:       {corr_xgb:.4f}")

#print(f"\n--- RIGOR DO ERRO (RMSE) ---")
#print(f"RMSE Random Forest: {rmse_rf:.2f} mm")
#print(f"RMSE XGBoost:       {rmse_xgb:.2f} mm")

import matplotlib.pyplot as plt
import seaborn as sns

#plt.figure(figsize=(10, 10))

# Plotando os pontos
#sns.scatterplot(x=y_test, y=rf_pred, color='blue', alpha=0.6, label='Random Forest')
#sns.scatterplot(x=y_test, y=xgb_pred, color='red', marker='x', alpha=0.6, label='XGBoost')

# Linha de Identidade (A perfeição: 45 graus)
lims = [0, max(y_test.max(), rf_pred.max())]
#plt.plot(lims, lims, color='black', linestyle='--', label='Identidade (Ideal)')

#plt.title('Gráfico de Dispersão: Alinhamento das Predições')
#plt.xlabel('Chuva Real (Observada em mm)')
#plt.ylabel('Chuva Prevista (Modelo em mm)')
#plt.legend()
#plt.grid(alpha=0.3)
#plt.axis('equal') # Garante que a escala 1:1 seja visualmente correta
#plt.show()

# Criando o Lag de 1 mês
df_ml['Chuva_Mes_Anterior'] = df_ml['Total'].shift(1)

# Como o primeiro mês da série não tem anterior, removemos a primeira linha (NaN)
df_ml = df_ml.dropna()

# Atualizando X e y com a nova variável
X = df_ml[['Anos', 'Meses', 'Chuva_Mes_Anterior']]
y = df_ml['Total']

# Nova divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Criando o Lag de 1 mês: O que aconteceu antes influencia o agora
df_ml['Chuva_Mes_Anterior'] = df_ml['Total'].shift(1)

# Removemos a primeira linha (que fica sem valor anterior)
df_ml = df_ml.dropna()

# Atualizando X (Features) com a nova variável
X = df_ml[['Anos', 'Meses', 'Chuva_Mes_Anterior']]
y = df_ml['Total']

# Nova divisão Treino/Teste (Mantendo a ordem cronológica)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)


import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Garantindo a Normalização (O "Datum" da rede)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Treinando a Rede Neural (Se ainda não foi feito)
mlp = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu',
                   max_iter=2000, random_state=42)
mlp.fit(X_train_scaled, y_train)

# 3. Gerando as Predições (Definindo mlp_pred)
mlp_pred = mlp.predict(X_test_scaled)

# 4. Agora sim, o Gráfico de Dispersão
plt.figure(figsize=(10, 10))

# Plotando os resultados
sns.scatterplot(x=y_test, y=mlp_pred, color='purple', alpha=0.7, label='Deep Learning (MLP)')

# Linha de Identidade (Referência Geométrica de Perfeição)
lims = [0, max(y_test.max(), mlp_pred.max())]
plt.plot(lims, lims, color='black', linestyle='--', linewidth=2, label='Ideal (Reta de 45°)')

plt.title('Ajuste de Mira: Deep Learning vs Realidade em Acaraú')
plt.xlabel('Chuva Real Observada (mm)')
plt.ylabel('Chuva Prevista pela Rede Neural (mm)')
plt.legend()
plt.grid(alpha=0.3)
plt.axis('equal')
plt.show()

# Exibindo métricas finais
print(f"RMSE Final: {np.sqrt(mean_squared_error(y_test, mlp_pred)):.2f} mm")



